# Pillar 3 — Big Data Explorer

Explores the outputs of `spark_pipeline.py` (5 Parquet tables) and `kafka_streaming.py` (Kafka run summary), right here in VS Code. Uses DuckDB's `read_parquet()` to query the Spark output directly — no pyarrow/pandas-parquet dependency needed.

Kernel: **Presight (Python 3.14)** — select it top-right if VS Code opens with a different one.

Run `spark_pipeline.py` and `kafka_streaming.py` first if the paths below don't exist yet (see `HOW_TO_RUN.md`).

In [ ]:
import json
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Parquet is a gitignored build artifact (outputs/artifacts/), separate from
# the tracked deliverables in outputs/results/ — see spark_pipeline.py.
SPARK_DIR = "../../../../outputs/artifacts/baiju_mohan/03_big_data/spark"
KAFKA_SUMMARY = "../../../../outputs/results/baiju_mohan/03_big_data/kafka/summary.json"

con = duckdb.connect(":memory:")


def read_table(name, partitioned=True):
    pattern = f"{SPARK_DIR}/{name}/**/*.parquet" if partitioned else f"{SPARK_DIR}/{name}/*.parquet"
    return con.sql(f"SELECT * FROM read_parquet('{pattern}')").df()

## project_activity_summary

In [ ]:
read_table("project_activity_summary").head(10)

## user_activity_summary

In [ ]:
read_table("user_activity_summary").head(10)

## escalation_log — resolved vs. unresolved, resolution time sanity check

In [ ]:
esc = read_table("escalation_log")
print(esc["resolved"].value_counts())
print("\nnegative resolution_time_hours (should be 0):", (esc.loc[esc["resolved"], "resolution_time_hours"] < 0).sum())
esc[esc["resolved"]].sort_values("resolution_time_hours").head(10)

## daily_event_volume — running total by event_type

In [ ]:
read_table("daily_event_volume").sort_values(["event_type", "event_date"]).head(15)

## peak_usage_analysis — top 20 date x hour

In [ ]:
read_table("peak_usage_analysis", partitioned=False)

## Kafka run summary

In [ ]:
with open(KAFKA_SUMMARY) as f:
    kafka_summary = json.load(f)

print(json.dumps(kafka_summary, indent=2))

In [ ]:
# Scratch cell — edit this query freely, e.g. any Spark table above
read_table("project_activity_summary").sort_values("escalation_count", ascending=False).head(10)